# Recipe Exploratory Data Analysis and Cleaning

Explore, validate and normalise the collected Lithuanian
recipe data. Prepare structured recipe and ingredient
datasets for PostgreSQL integration and AI functionality.

## 1. Setup and Data Loading

In [ ]:
import json
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw/recipes_jaukus")
PROCESSED_DIR = Path("../data/processed")

In [ ]:
with open(RAW_DIR / "recipes.json", encoding="utf-8") as f:
    recipes_raw = json.load(f)

print("Raw recipes:", len(recipes_raw))
print("Unique recipes:", len({r["id"] for r in recipes_raw}))

## 2. Data Extraction

Transform nested recipe JSON into separate recipe-level
and ingredient-level DataFrames using the extraction
functions developed during API exploration.

In [ ]:
def extract_recipe(data):
    content = data["content"]
    lt = content["lt"]
    nutrition = content.get("nutrition") or {}

    return {
        "recipe_id": data["id"],
        "slug": data["slug"],
        "title": lt["title"],
        "description": lt.get("description"),
        "servings": content.get("servings"),
        "prep_minutes": content.get("prepMinutes"),
        "cook_minutes": content.get("cookMinutes"),
        "cuisine": content.get("cuisine"),
        "categories": [
            c["nameLt"] for c in data.get("categories", [])
        ],
        "instructions": [
            step["text"]
            for step in lt.get("steps", [])
            if step.get("text")
        ],
        "kcal": nutrition.get("kcal"),
        "protein": nutrition.get("protein"),
        "fat": nutrition.get("fat"),
        "carbohydrates": nutrition.get("carbohydrates"),
        "sugars": nutrition.get("sugars"),
        "nutrition_complete": nutrition.get("complete"),
        "url": f"https://jaukusskoniai.lt/r/{data['slug']}"
    }

In [ ]:
def extract_ingredients(data):
    rows = []

    for position, ingredient in enumerate(
        data["content"].get("ingredients", [])
    ):
        original = ingredient.get("original") or {}

        quantity = ingredient.get("quantity")
        unit = ingredient.get("unit")

        conversion = next(
            (
                c for c in (ingredient.get("units") or [])
                if c["unit"] == unit
            ),
            {}
        )

        grams_per_unit = conversion.get("gramsPerUnit")

        total_grams = (
            quantity * grams_per_unit
            if quantity is not None and grams_per_unit is not None
            else None
        )

        rows.append({
            "recipe_id": data["id"],
            "position": position,
            "ingredient_id": ingredient.get("ingredientId"),
            "name_lt": ingredient.get("nameLt"),
            "original_name": original.get("name"),
            "notes": original.get("notes"),
            "group": original.get("group"),
            "quantity": quantity,
            "unit": unit,
            "default_unit": ingredient.get("defaultUnit"),
            "grams_per_unit": grams_per_unit,
            "total_grams": total_grams,
            "conversion_source": conversion.get("source"),
            "source_category": ingredient.get("sourceCategory"),
            "needs_review": ingredient.get("needsReview")
        })

    return rows

In [ ]:
recipe_rows = []
ingredient_rows = []

for recipe in recipes_raw:
    recipe_rows.append(extract_recipe(recipe))
    ingredient_rows.extend(extract_ingredients(recipe))

df_recipes = pd.DataFrame(recipe_rows)
df_ingredients = pd.DataFrame(ingredient_rows)

print("Recipes shape:", df_recipes.shape)
print("Ingredients shape:", df_ingredients.shape)

In [ ]:
display(df_recipes.head())
display(df_ingredients.head())

## 3. Initial Data Quality Assessment

Evaluate dataset completeness, identifier uniqueness,
referential integrity and ingredient conversion coverage
before applying cleaning rules.

In [ ]:
display(
    df_recipes.isna().sum()
    .sort_values(ascending=False)
    .to_frame("Missing values")
)

display(
    df_ingredients.isna().sum()
    .sort_values(ascending=False)
    .to_frame("Missing values")
)

In [ ]:
print("Total recipes:", len(df_recipes))

print("Duplicate recipe IDs:",
      df_recipes["recipe_id"].duplicated().sum())

print("Total ingredient entries:", len(df_ingredients))

print("Unique ingredient IDs:",
      df_ingredients["ingredient_id"].nunique())

print("Duplicate ingredient entries:",
      df_ingredients.duplicated(
          subset=["recipe_id", "position"]
      ).sum())

print("Orphan ingredient entries:",
      (~df_ingredients["recipe_id"].isin(
          df_recipes["recipe_id"]
      )).sum())

print("Recipes without ingredients:",
      (~df_recipes["recipe_id"].isin(
          df_ingredients["recipe_id"]
      )).sum())

In [ ]:
print("Missing quantities:",
      df_ingredients["quantity"].isna().sum())

print("Non-positive quantities:",
      (df_ingredients["quantity"] <= 0).sum())

print("Missing weight conversions:",
      df_ingredients["total_grams"].isna().sum())

print("Ingredients requiring review:",
      df_ingredients["needs_review"].eq(True).sum())

print("\nMeasurement units:")
print(df_ingredients["unit"].value_counts())

print("\nConversion sources:")
print(df_ingredients["conversion_source"].value_counts())

## 4. Exploratory Data Analysis

Explore recipe diversity, ingredient frequency, nutritional
characteristics and potential ingredient quality issues.

In [ ]:
display(
    df_ingredients.loc[
        df_ingredients["needs_review"].eq(True),
        [
            "name_lt",
            "original_name",
            "notes",
            "quantity",
            "unit",
            "total_grams"
        ]
    ]
)

In [ ]:
ingredient_frequency = (
    df_ingredients
    .groupby(["ingredient_id", "name_lt"])["recipe_id"]
    .nunique()
    .sort_values(ascending=False)
)

ingredient_frequency.head(20)

In [ ]:
import matplotlib.pyplot as plt

ingredient_frequency.droplevel("ingredient_id").head(20).sort_values().plot(
    kind="barh",
    figsize=(10, 7)
)

plt.title("Top 20 most frequently used ingredients")
plt.xlabel("Number of recipes")
plt.ylabel("Ingredient")
plt.tight_layout()
plt.show()

In [ ]:
category_counts = (
    df_recipes["categories"]
    .explode()
    .value_counts()
)

category_counts

In [ ]:
category_counts.head(20).sort_values().plot(
    kind="barh",
    figsize=(10, 7)
)

plt.title("Recipe category distribution")
plt.xlabel("Number of recipes")
plt.tight_layout()
plt.show()

In [ ]:
df_recipes[
    [
        "servings",
        "prep_minutes",
        "cook_minutes",
        "kcal",
        "protein",
        "fat",
        "carbohydrates",
        "sugars"
    ]
].describe().round(2)

In [ ]:
print("Non-positive servings:",
      (df_recipes["servings"] <= 0).sum())

print("Recipes with zero preparation and cooking time:",
      (
          (df_recipes["prep_minutes"] == 0) &
          (df_recipes["cook_minutes"] == 0)
      ).sum())

print("Recipes without instructions:",
      df_recipes["instructions"].apply(len).eq(0).sum())

print("Incomplete nutritional information:",
      df_recipes["nutrition_complete"].eq(False).sum())

print("Non-positive weight conversions:",
      (df_ingredients["total_grams"] <= 0).sum())

## 5. Data Cleaning and Normalisation

Prepare recipe and ingredient datasets for database
integration. Calculate per-serving nutritional values,
standardise equivalent measurement units and preserve
data quality indicators.

In [ ]:
df_recipes_clean = df_recipes.copy()
df_ingredients_clean = df_ingredients.copy()

In [ ]:
nutrition_columns = [
    "kcal",
    "protein",
    "fat",
    "carbohydrates",
    "sugars"
]

for column in nutrition_columns:
    df_recipes_clean[f"{column}_per_serving"] = (
        df_recipes_clean[column] /
        df_recipes_clean["servings"]
    ).round(2)

In [ ]:
df_recipes_clean["total_minutes"] = (
    df_recipes_clean["prep_minutes"] +
    df_recipes_clean["cook_minutes"]
)

df_recipes_clean["time_unknown"] = (
    df_recipes_clean["total_minutes"] == 0
)

In [ ]:
df_ingredients_clean["unit"] = (
    df_ingredients_clean["unit"]
    .replace({"piece": "pcs"})
)

In [ ]:
df_ingredients_clean["quantity_estimated"] = (
    df_ingredients_clean["conversion_source"] == "estimate"
)

In [ ]:
df_ingredients_clean.loc[
    df_ingredients_clean["needs_review"].eq(True),
    "name_lt"
].value_counts()

In [ ]:
df_recipes_clean[
    [
        "kcal_per_serving",
        "protein_per_serving",
        "fat_per_serving",
        "carbohydrates_per_serving",
        "sugars_per_serving",
        "total_minutes"
    ]
].describe().round(2)

## 6. Final Validation and Parquet Export

Validate the processed recipe and ingredient datasets
and export them as Parquet files for subsequent
PostgreSQL integration.

In [ ]:
assert df_recipes_clean["recipe_id"].is_unique

assert not df_ingredients_clean.duplicated(
    subset=["recipe_id", "position"]
).any()

assert df_ingredients_clean["recipe_id"].isin(
    df_recipes_clean["recipe_id"]
).all()

assert (df_recipes_clean["servings"] > 0).all()

assert (df_ingredients_clean["quantity"] > 0).all()

assert (df_ingredients_clean["total_grams"] > 0).all()

print("All validation checks passed.")

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_recipes_clean.to_parquet(
    PROCESSED_DIR / "recipes.parquet",
    index=False
)

df_ingredients_clean.to_parquet(
    PROCESSED_DIR / "recipe_ingredients.parquet",
    index=False
)

print("Recipe datasets successfully exported.")

In [ ]:
for path in PROCESSED_DIR.glob("*.parquet"):
    print(path.name, pd.read_parquet(path).shape)